In [ ]:
import torch
import torch.nn as nn
import zipfile
import gdown
import os
from torch.utils.data import Subset, DataLoader, Dataset
from sklearn.model_selection import train_test_split
from torchvision.transforms import Resize
from torchvision.io import read_image
import matplotlib.pyplot as plt
from MLP import *
from torchsummary import summary

In [ ]:
Folder = 'data/'
ID_FILE = '1MiNzagnzW-scHQgx3Mku1KUuPaJQ4TBQ'
FILE_NAME = 'FER-2013.zip'

In [ ]:
os.makedirs(Folder,exist_ok=True)

# Tải DL và giải nén DL

In [ ]:
gdown.download(id = ID_FILE, output= FILE_NAME)

In [ ]:
# Giải nén file FER-2013.zip
with zipfile.ZipFile('FER-2013.zip','r') as zip_ref:
    zip_ref.extractall(Folder)

In [ ]:

train_dir = 'data/train'
test_dir = 'data/test'

# Thiêt lập 2 dictionary để ánh xạ từ nhãn -> index và từ index -> nhãn
classes = os.listdir(train_dir)
print(classes)
label2idx = {label:idx for idx, label in enumerate(classes)}
print(label2idx)
idx2label = {idx:label for label, idx in label2idx.items()}
print(idx2label)

In [ ]:
Dataset??

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, img_dir:str, train: bool = None, normalize: bool = None, label2idx = None, trainsplit :float = 0.8, img_size: int = 28):
        self.img_dir = img_dir # Tên thư mục ảnh đang xét
        self.train = train # Đây là DL tập train
        self.normalize = normalize # Có chuẩn hóa ảnh không
        self.label2idx = label2idx  # Thư mục mã hóa từ nhãn sang chỉ số
        self.trainsplit = trainsplit # Tỷ lệ của tập train
        self.img_paths, self.img_labels = self.read_img_files() # Lấy toàn bộ đường dẫn của tát cả các ảnh và nhãn tương ứng của chúng
        self.resize = Resize((img_size,img_size))
        if self.train is not None and (self.train or  not self.train):
            train_data, val_data = train_test_split(
                list(zip(self.img_paths, self.img_labels)),
                train_size=trainsplit,
                # Tham số stratify -> giúp giữ nguyên phân bố của nhãn khi chia
                stratify=self.img_labels
                )
            if self.train:
                self.img_paths, self.img_labels = zip(*train_data)
            else:
                self.img_paths, self.img_labels = zip(*val_data)
    def read_img_files(self):
        img_paths = [] # List đường  dẫn của ảnh
        img_labels = [] # list nhãn tương ứng của ảnh
        for clss in label2idx.keys():
            # Xét mỗi ảnh trong mỗi thư mục nhãn 
            for img in os.listdir(os.path.join(self.img_dir, clss)):
                # Lấy đường dẫn từ thư mục gốc -> thư mục nhãn -> ảnh
                img_paths.append(os.path.join(self.img_dir, clss, img))
                img_labels.append(clss)
        return img_paths, img_labels
    # Bắt buộc overwrite - phải có khi kế thừa lớp Dataset của pytorch
    def __getitem__(self, index):
        img_path = self.img_paths[index]
        img_label = self.img_labels[index]
        # Đặt lại kích thước của mỗi ảnh sao cho tất cả các ảnh có cùng kích thước
        img = self.resize(read_image(path=img_path))
        # chuyển kiểu Dl của ảnh sang dạng torch.float32
        img = img.float()
        if self.normalize:
            img = (img/127.5) - 1
        label = label2idx[img_label]
        return img, label
    # Bắt buộc overwrite - phải có khi kế thừa lớp Dataset của Pytorch
    def __len__(self):
        return len(self.img_paths)

In [ ]:
dataset_train  = ImageDataset(img_dir=train_dir,label2idx=label2idx, normalize=True)
image, lable = dataset_train[0]
image.shape

In [ ]:
# Tập test riêng (thư mục data/test), không trùng với phần train/validation
dataset_test = ImageDataset(img_dir=test_dir, train=None, label2idx=label2idx, normalize=True)

In [ ]:
# Hiển thị 5 ảnh đầu tiên và nhãn tương ứng
fig, axes = plt.subplots(1, 5, figsize=(14, 3))
for i in range(5):
    img, label_idx = dataset_train[i]
    # Ảnh đã normalize: (img/127.5)-1  ->  khôi phục về [0, 255] để hiển thị
    img_np = ((img + 1) * 127.5).clamp(0, 255).byte().permute(1, 2, 0).numpy()
    axes[i].imshow(img_np, cmap = 'gray')
    axes[i].set_title(idx2label[label_idx])
    axes[i].axis('off')
plt.tight_layout()
plt.show()

# Xây dựng và khởi tạo mô hình MLP cho bài toán phân loại ảnh

In [ ]:
BaseMLP??

In [ ]:
class MLPclassifier(BaseMLP):
    def __init__(self):
        super().__init__()
    def predict(self, X):
        with torch.no_grad():
            logits = self.forward(X)
            return torch.argmax(logits,dim =1)
    def get_accuracy(self, logits, y):
        try:
            return torch.mean((torch.argmax(logits,dim =1) == y).float())
        except:
            return torch.mean((torch.argmax(logits,dim =1) == torch.argmax(y, dim =1)).float())
    def compute_loss(self, logits, y):
        return self.criterion(logits,y)

In [ ]:
classifier = MLPclassifier()

In [ ]:
input_dims = img.shape[1]*img.shape[2]
print(input_dims)
output_dims = len(classes)
print(output_dims)

# Xây dựng các Layer cho mô hình MultiLayer Perceptron

In [ ]:
# Layer 1 - flatten -> duỗi ảnh thành 1 vector
classifier.Add_layer(nn.Flatten())

# layer 2 
classifier.Add_layer(
    nn.Linear(in_features=input_dims, out_features=256)
)
# layer 3 - activation function
classifier.Add_layer(
    nn.GELU()
)
# Layer 4
classifier.Add_layer(
    nn.Linear(in_features=256,out_features=output_dims)
)

In [ ]:
summary(classifier.model, input_size=(1,28,28),batch_size=512, device='cpu')

In [ ]:
# Huấn luyện mô hình 
classifier.fit(dataset=dataset_train,n_epochs=100, optimizer='Adam', batch_size=1024, criterion='CE',validation_split=0.2, verbose=2, is_shuffle=True)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epochs = range(1, len(classifier.Losses) + 1)

axes[0].plot(epochs, classifier.Losses, label="Train")
if classifier.Val_Losses:
    axes[0].plot(epochs, classifier.Val_Losses, label="Validation")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss theo epoch (train / validation)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, classifier.Accuracies, label="Train")
if classifier.Val_Accuracies:
    axes[1].plot(epochs, classifier.Val_Accuracies, label="Validation")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Accuracy theo epoch (train / validation)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()